# Module 3 — Introduction to Deep Learning

## Notebook 2: Training, Generalisation, and Architecture Choice

This guided notebook examines what happens while a neural network trains. You
will compare small and larger models, use validation data to recognise
overfitting, apply regularisation and early stopping, simulate gradient
multiplication, and select an architecture that matches a data structure.

### Learning outcomes

By the end, you should be able to:

- distinguish training, validation, and test data;
- recognise underfitting, useful fit, and overfitting from learning curves;
- explain how regularisation and early stopping can improve generalisation;
- describe why repeated gradient multiplication can vanish or explode;
- select CNNs, RNNs, or Transformers based on the structure of an input.

Run the notebook from top to bottom. All data is generated locally with a fixed
random seed; no internet connection or GPU is required.

> **Advanced track:** This is the same workshop as the beginner track. Complete each `TODO` before running the dependent cells.


In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np

from sklearn.datasets import make_moons
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier

RANDOM_SEED = 42
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True

## 1. Training, validation, and test data

A model learns its parameters from the **training set**. We use a separate
**validation set** while making design choices, such as choosing model size or
regularisation. The **test set** is held back until the end to estimate how
well the final chosen model generalises to unseen examples.

Using three sets helps prevent us from repeatedly adjusting a model until it
looks good only on the examples we already inspected.

In [ ]:
# Generate a noisy curved classification problem.
X, y = make_moons(
    n_samples=360,
    noise=0.30,
    random_state=RANDOM_SEED,
)

# First reserve 25% of all examples as an untouched final test set.
X_development, X_test, y_development, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=RANDOM_SEED,
    stratify=y,
)

# Split the remaining development data into training and validation data.
X_train, X_validation, y_train, y_validation = train_test_split(
    X_development,
    y_development,
    test_size=1 / 3,
    random_state=RANDOM_SEED,
    stratify=y_development,
)

print(f"Training examples:   {len(X_train)}")
print(f"Validation examples: {len(X_validation)}")
print(f"Test examples:       {len(X_test)}")

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharex=True, sharey=True)
for axis, X_part, y_part, title in zip(
    axes,
    [X_train, X_validation, X_test],
    [y_train, y_validation, y_test],
    ["Training set", "Validation set", "Final test set"],
):
    axis.scatter(
        X_part[:, 0],
        X_part[:, 1],
        c=y_part,
        cmap="coolwarm",
        edgecolor="black",
        alpha=0.85,
    )
    axis.set_title(title)
    axis.set_xlabel("Feature 1")

axes[0].set_ylabel("Feature 2")
plt.tight_layout()
plt.show()

## 2. Learning curves: small and larger neural networks

A learning curve shows how a model performs as training continues.

- **Underfitting:** both training and validation accuracy remain low.
- **Useful fit:** training and validation accuracy improve and remain similar.
- **Overfitting:** training accuracy continues to improve while validation
  accuracy stops improving or decreases.

The helper below trains one epoch at a time. Its implementation is supplied:
you only need to interpret the resulting curves.

In [ ]:
def train_with_history(hidden_layer_sizes, alpha, epochs=100):
    """Train a small MLP one epoch at a time and record two accuracies."""

    # TODO:
    # Create an MLPClassifier that:
    # - uses `hidden_layer_sizes` and `alpha` from this function's arguments;
    # - uses ReLU activation;
    # - uses stochastic gradient descent: solver="sgd";
    # - trains one epoch at a time: max_iter=1;
    # - retains learned weights between calls to fit: warm_start=True;
    # - uses RANDOM_SEED for reproducibility.
    model = ...

    training_accuracy = []
    validation_accuracy = []

    # max_iter=1 intentionally produces a convergence warning after each epoch.
    # The warning is expected here because we are controlling epochs ourselves.
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")

        for _ in range(epochs):
            model.fit(X_train, y_train)

            # TODO:
            # Record accuracy for the training data and validation data.
            training_accuracy.append(...)
            validation_accuracy.append(...)

    return model, np.array(training_accuracy), np.array(validation_accuracy)


# TODO:
# Train a small model with one hidden layer containing two neurons.
# Use alpha=0.001.
small_model, small_train, small_validation = train_with_history(
    hidden_layer_sizes=...,
    alpha=...,
)

# TODO:
# Train a larger model with two hidden layers containing fifty neurons each.
# Use alpha=0.00001.
large_model, large_train, large_validation = train_with_history(
    hidden_layer_sizes=...,
    alpha=...,
)

epochs = np.arange(1, len(small_train) + 1)

plt.plot(epochs, small_train, label="Small model — training", linewidth=2)
plt.plot(
    epochs,
    small_validation,
    label="Small model — validation",
    linestyle="--",
    linewidth=2,
)
plt.plot(epochs, large_train, label="Large model — training", linewidth=2)
plt.plot(
    epochs,
    large_validation,
    label="Large model — validation",
    linestyle="--",
    linewidth=2,
)
plt.ylim(0.35, 1.03)
plt.xlabel("Training epoch")
plt.ylabel("Accuracy")
plt.title("Training and validation learning curves")
plt.legend()
plt.show()

print(f"Small-model final training accuracy:   {small_train[-1]:.3f}")
print(f"Small-model final validation accuracy: {small_validation[-1]:.3f}")
print(f"Large-model final training accuracy:   {large_train[-1]:.3f}")
print(f"Large-model final validation accuracy: {large_validation[-1]:.3f}")

### Interpreting the curves

The small model may underfit because it has only two hidden neurons. The larger
model often achieves higher training accuracy, but its validation accuracy may
not improve by the same amount. The difference between training and validation
performance is called the **generalisation gap**.

Your exact values are reproducible because the random seed is fixed. Focus on
the relationship between the two curves rather than expecting one particular
accuracy threshold.

In [ ]:
def plot_decision_boundary(model, X_data, y_data, title):
    """Plot predicted class regions for a fitted two-feature classifier."""
    padding = 0.4
    x_min, x_max = X_data[:, 0].min() - padding, X_data[:, 0].max() + padding
    y_min, y_max = X_data[:, 1].min() - padding, X_data[:, 1].max() + padding

    x_grid, y_grid = np.meshgrid(
        np.linspace(x_min, x_max, 250),
        np.linspace(y_min, y_max, 250),
    )
    grid_points = np.column_stack([x_grid.ravel(), y_grid.ravel()])
    predicted = model.predict(grid_points).reshape(x_grid.shape)

    plt.contourf(x_grid, y_grid, predicted, alpha=0.25, cmap="coolwarm")
    plt.scatter(
        X_data[:, 0],
        X_data[:, 1],
        c=y_data,
        cmap="coolwarm",
        edgecolor="black",
        alpha=0.85,
    )
    plt.xlabel("Feature 1")
    plt.ylabel("Feature 2")
    plt.title(title)


plot_decision_boundary(
    small_model,
    X_validation,
    y_validation,
    "Small model on validation data",
)
plt.show()

plot_decision_boundary(
    large_model,
    X_validation,
    y_validation,
    "Larger model on validation data",
)
plt.show()

## 3. Reducing overfitting: regularisation and early stopping

Two common strategies can reduce overfitting:

- **Regularisation** discourages unnecessarily large weights and overly complex
  solutions. In `MLPClassifier`, `alpha` controls the amount of L2
  regularisation.
- **Early stopping** monitors an internal validation portion of the training
  data and stops when validation performance has not improved for several
  iterations.

Neither strategy guarantees a better model in every situation. They are
tools for balancing model capacity against generalisation.

In [ ]:
# TODO:
# Create and fit a larger model that uses both regularisation and early stopping.
#
# Requirements:
# - two hidden layers of fifty neurons: hidden_layer_sizes=(50, 50)
# - ReLU activation
# - Adam optimiser: solver="adam"
# - L2 regularisation: alpha=0.02
# - early_stopping=True
# - validation_fraction=0.20
# - n_iter_no_change=20
# - max_iter=500
# - deterministic seed: random_state=RANDOM_SEED
regularised_model = ...
regularised_model.fit(...)

models_to_compare = {
    "Small model": small_model,
    "Larger model": large_model,
    "Regularised + early stopping": regularised_model,
}

for name, model in models_to_compare.items():
    train_accuracy = accuracy_score(y_train, model.predict(X_train))
    validation_accuracy = accuracy_score(
        y_validation,
        model.predict(X_validation),
    )
    print(
        f"{name:30s} "
        f"training={train_accuracy:.3f} "
        f"validation={validation_accuracy:.3f}"
    )

print(
    "\nEarly-stopped model iterations:",
    regularised_model.n_iter_,
)

plot_decision_boundary(
    regularised_model,
    X_validation,
    y_validation,
    "Regularised model with early stopping on validation data",
)
plt.show()

## 4. Final evaluation on the test set

Choose a model using training and validation evidence. Only then use the test
set once. In this example, we evaluate the regularised, early-stopped model on
the untouched test data.

In [ ]:
final_test_predictions = regularised_model.predict(X_test)
final_test_accuracy = accuracy_score(y_test, final_test_predictions)

print(
    "Final test accuracy for the regularised, early-stopped model:",
    f"{final_test_accuracy:.3f}",
)

plot_decision_boundary(
    regularised_model,
    X_test,
    y_test,
    f"Final test set (accuracy: {final_test_accuracy:.2f})",
)
plt.show()

## 5. Vanishing and exploding gradients

During backpropagation, gradients pass backwards through many layers. At a
high level, this repeatedly multiplies values related to local derivatives and
weights.

If typical factors are below one, repeated multiplication can make a gradient
very small: a **vanishing gradient**. If factors are above one, it can become
very large: an **exploding gradient**.

The calculation below is a numerical intuition, not a full implementation of
backpropagation.

In [ ]:
depth = np.arange(1, 51)
starting_gradient = 1.0

# Each line represents repeatedly multiplying by a typical factor.
factors = {
    "shrinking factor 0.5": 0.5,
    "slightly shrinking factor 0.9": 0.9,
    "stable factor 1.0": 1.0,
    "slightly growing factor 1.1": 1.1,
    "growing factor 1.5": 1.5,
}

for label, factor in factors.items():
    # TODO:
    # Calculate the gradient magnitude after each repeated multiplication.
    # Start at `starting_gradient` and repeatedly multiply by `factor`.
    gradient_size = ...
    plt.semilogy(depth, gradient_size, label=label, linewidth=2)

plt.xlabel("Number of repeated multiplications")
plt.ylabel("Gradient magnitude (logarithmic scale)")
plt.title("Why gradients can vanish or explode")
plt.legend()
plt.show()

for label, factor in factors.items():
    # TODO:
    # Calculate the final gradient magnitude after the last multiplication.
    final_gradient = ...
    print(f"{label:32s} final magnitude = {final_gradient:.3e}")

## 6. Choosing an architecture for the data structure

Different neural-network architectures are useful for different kinds of
input. The best choice depends on the data structure and task, not on a simple
ranking of architectures.

| Input structure | Often suitable starting point | Why |
| --- | --- | --- |
| Image or spatial grid | CNN | Nearby pixels and local patterns matter. |
| Short ordered time series | RNN or similar sequence model | Order and recent state matter. |
| Sentence or document | Transformer | Attention can relate tokens across a sequence. |
| Long-context text or multimodal sequence | Transformer | Attention can connect distant relevant elements. |

This workshop does **not** train these architectures. The activity is about
matching an architecture family to the structure of the input.

In [ ]:
# TODO:
# Complete each tuple with:
# 1. a suitable architecture family;
# 2. a short reason based on the input structure.
#
# Choose from CNN, RNN (or another sequence model), and Transformer.
architecture_cases = [
    (
        "Classify defects in photographs of manufactured parts",
        ...,
        ...,
    ),
    (
        "Estimate tomorrow's energy demand from hourly measurements",
        ...,
        ...,
    ),
    (
        "Translate a sentence while relating words that may be far apart",
        ...,
        ...,
    ),
    (
        "Answer questions using a long technical report as context",
        ...,
        ...,
    ),
]

for task, architecture, reason in architecture_cases:
    print(f"Task: {task}")
    print(f"Suitable starting point: {architecture}")
    print(f"Reason: {reason}\n")

## 7. Summary and exit ticket

### Key ideas

- Training data fits model parameters; validation data supports model choices;
  test data estimates final generalisation.
- A large gap between training and validation performance can indicate
  overfitting.
- Regularisation and early stopping are two ways to reduce overfitting.
- Repeated multiplication by values smaller than one can vanish; repeated
  multiplication by values larger than one can explode.
- CNNs, RNNs, and Transformers are architecture choices guided by the input
  structure and task.

### Exit ticket

Answer these questions in your own words:

1. Why should the test set not guide repeated model changes?
2. Which curve pattern suggests overfitting, and why?
3. How can early stopping help a larger neural network?
4. What happens to a gradient after many multiplications by $$0.9$$? What
   happens after many multiplications by $$1.1$$?
5. Choose an architecture family for:  
   a. satellite images;  
   b. daily rainfall measurements;  
   c. a long legal document.  
   Explain each choice using the data structure.